<a href="https://colab.research.google.com/github/harpqbrace/thegridcitypress/blob/main/module_02_assignment_02_local_model_task_portfolio_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ### Note on Labs and Assignments:
>
> 🔧 Look for the **wrench emoji** — it marks code you must change. Routine run-only cells do not use it.
>
> 🖊 Look for the **writing emoji** — it marks analysis you must write.
>
> These sections are graded and are not optional.
>

# Module 2 Assignment 2: Local Model Task Portfolio

**Notebook:** Student Template  
**Student:** Edit the configuration cell  
**Required models:** `gemma3:1b`, `gemma3:4b`, `llama3.2:1b`, and `llama3.2:3b`

This notebook supports the complete Assignment 2 workflow: four direct business
tasks, prompt revision, a controlled four-model comparison, evidence-based scoring, reflection, and AI-use disclosure.


## Student Introduction: What is this Assignment Is About?

This assignment asks you to work directly with small language models running on your own machine via Ollama. You will practice two core skills:

1. **Prompt engineering** — writing and iteratively improving instructions that guide a model toward a useful business output.
2. **Model evaluation** — systematically comparing four models on the same task and scoring them with evidence.

**What you will produce:**

- **Part 1:** Four business tasks (extraction, summarization, drafting, classification). For each task, you write an initial zero-shot instruction, run it, diagnose a specific weakness in the output, revise the instruction using a named prompting strategy, run it again, and evaluate the improvement.
- **Part 2:** A controlled four-model comparison using a shared service-request packet. You design a single instruction, run it on all four models without changing anything, then score each model across four dimensions with evidence from their outputs.
- **Part 3:** A 350–500 word reflection answering seven specific questions about what you observed.

**Before you start:**
1. Run the initial code blocks to install Ollama and start it.
2. Replace `"Your Name"` in the configuration cell below with your actual name.
3. Run all cells from top to bottom in order.

The notebook will raise an error and stop if any required `TODO` is still present when you try to run a model — this is intentional so you do not accidentally submit incomplete work.

## Important Instructions

1. Read the assignment before editing this notebook.
2. Edit only cells marked for student work.
3. Do not change the comparison source packet, model list, or shared settings.
4. Preserve the first output from every run.
5. Before submitting, restart the kernel and run all cells from top to bottom.

The template intentionally raises a clear error when a required `TODO` remains.


## Setup Ollama

This notebook will download, install and start [Ollama](https://ollama.com/download). The four default model downloads require approximately 8 GB in total.





### What is Ollama?

Ollama is a tool that lets you run AI language models on your own computer instead of only using an online service like ChatGPT.

For a beginner, you can think of it as a local “AI model manager.” It helps you download a model, start it, and send it prompts. For example, instead of calling an online API from OpenAI, Google, or Anthropic, you can call an Ollama model running on your laptop or server.

The basic idea is:



*   You install Ollama.
*   You download a model, such as Llama, Gemma, or Mistral.
*   You send text to the model.
*  The model sends text back.


Why are we doing this rather than using Claude or ChatGPT?

* Reproducability. The notebook allows students to all follow the same steps and instructions.
* Cost: API access to Anthropic,OpenAI, Google models is not free. These models are free to run. The trade-off? (There always is one).
* What do we sacrifice for using the free models? Quality of responses and speed. We'll be using CPUs since these models are small language models (SLMs) rather than large language models (LLMs)






In [3]:
import subprocess
import time

In [4]:
# Download and install Ollama (Google Colab only — skip if running locally)
install_zstd = subprocess.run(
    "sudo apt-get install zstd",
    shell=True,
    capture_output=True,
    text=True,
)

install_zstd

CompletedProcess(args='sudo apt-get install zstd', returncode=0, stdout='Reading package lists...\nBuilding dependency tree...\nReading state information...\nThe following NEW packages will be installed:\n  zstd\n0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.\nNeed to get 603 kB of archives.\nAfter this operation, 1,695 kB of additional disk space will be used.\nGet:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]\nFetched 603 kB in 0s (5,354 kB/s)\nSelecting previously unselected package zstd.\n(Reading database ... \n(Reading database ... 5%\n(Reading database ... 10%\n(Reading database ... 15%\n(Reading database ... 20%\n(Reading database ... 25%\n(Reading database ... 30%\n(Reading database ... 35%\n(Reading database ... 40%\n(Reading database ... 45%\n(Reading database ... 50%\n(Reading database ... 55%\n(Reading database ... 60%\n(Reading database ... 65%\n(Reading database ... 70%\n(Reading database ... 75%\n(Reading data

In [5]:
# RUN THIS CELL. YOU SHOULD SEE Ollama installed and Ollama server is running messages.

# Download and install Ollama
install = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    capture_output=True,
    text=True,
)
if install.returncode != 0:
    raise RuntimeError(f"Ollama installation failed:\n{install.stderr}")
print("Ollama installed.")

# Start the Ollama server as a background process
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Give the server a few seconds to initialize before any requests are made
time.sleep(3)
print("Ollama server is running.")

Ollama installed.
Ollama server is running.


In [6]:
# RUN THIS CELL

from datetime import datetime
from hashlib import sha256
from time import perf_counter
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json
import textwrap

from IPython.display import Markdown, display
import subprocess
import time


REFERENCE_MODE = False
AUTO_PULL_MODELS = True
OLLAMA_BASE_URL = "http://localhost:11434"

REQUIRED_MODELS = [
    "gemma3:1b",
    "gemma3:4b",
    "llama3.2:1b",
    "llama3.2:3b",
]
BASELINE_MODEL = "gemma3:1b"
GENERATION_OPTIONS = {
    "temperature": 1,
    #"seed": 4490,
    "num_ctx": 8192,
    "num_predict": 900,
}




In [7]:
# RUN THIS CELL

REFERENCE_OUTPUTS = {}


def require_finished(label, value):
    """Stop before a model run when a required student field is unfinished."""
    if value is None or "TODO" in str(value):
        raise ValueError(f"Complete {label} before running this cell.")


def ollama_request(path, payload=None, timeout=120):
    """Send a JSON request to the local Ollama service."""
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        f"{OLLAMA_BASE_URL}{path}",
        data=data,
        headers={"Content-Type": "application/json"},
        method="GET" if payload is None else "POST",
    )
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        details = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Ollama returned HTTP {exc.code}: {details}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(
            "Cannot connect to Ollama at http://localhost:11434. "
            "Install and start Ollama, then rerun this cell."
        ) from exc


def chat_once(model, prompt, run_key):
    """Run one independent prompt and return content plus observable metadata."""
    if REFERENCE_MODE:
        fixture = REFERENCE_OUTPUTS[run_key]
        return {
            "model": model,
            "run_key": run_key,
            "recorded_at": fixture["recorded_at"],
            "elapsed_seconds": fixture["elapsed_seconds"],
            "content": fixture["content"],
            "prompt_eval_count": None,
            "eval_count": None,
            "reference_fixture": True,
        }

    started_at = datetime.now().astimezone().isoformat(timespec="seconds")
    start = perf_counter()
    response = ollama_request(
        "/api/chat",
        {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "keep_alive": 0,
            "options": GENERATION_OPTIONS,
        },
        timeout=900,
    )

    #print(f"Generation Options:{GENERATION_OPTIONS}")

    elapsed = round(perf_counter() - start, 2)
    return {
        "model": model,
        "run_key": run_key,
        "recorded_at": started_at,
        "elapsed_seconds": elapsed,
        "content": response["message"]["content"].strip(),
        "prompt_eval_count": response.get("prompt_eval_count"),
        "eval_count": response.get("eval_count"),
        "reference_fixture": False,
    }


def display_record(record):
    metadata = (
        f"**Model:** `{record['model']}`  \n"
        f"**Recorded:** {record['recorded_at']}  \n"
        f"**Elapsed:** {record['elapsed_seconds']} seconds"
    )
    display(Markdown(metadata))
    display(Markdown(record["content"]))


def compose_prompt(instruction, source):
    return (
        instruction.strip()
        + "\n\nSOURCE\n------\n"
        + source.strip()
    )


def run_part1(task_key, stage, instruction, source):
    require_finished(f"{task_key} {stage} instruction", instruction)
    return chat_once(
        BASELINE_MODEL,
        compose_prompt(instruction, source),
        f"part1_{task_key}_{stage}",
    )


In [8]:
# RUN THIS CELL
# You should see
# Attempting to pull model: gemma3:1b
# Successfully pulled gemma3:1b.
# etc.

# Check and pull required models if AUTO_PULL_MODELS is True
if AUTO_PULL_MODELS:
    print(f"Checking and pulling required models: {REQUIRED_MODELS}")
    for model_name in REQUIRED_MODELS:
        print(f"Attempting to pull model: {model_name}")
        # Use subprocess to run ollama pull command
        pull_result = subprocess.run(
            f"ollama pull {model_name}",
            shell=True,
            capture_output=True,
            text=True,
        )
        if pull_result.returncode != 0:
            print(f"Failed to pull {model_name}:\n{pull_result.stderr}")
        else:
            print(f"Successfully pulled {model_name}.")
        time.sleep(1) # Give a moment between pulls

Checking and pulling required models: ['gemma3:1b', 'gemma3:4b', 'llama3.2:1b', 'llama3.2:3b']
Attempting to pull model: gemma3:1b
Successfully pulled gemma3:1b.
Attempting to pull model: gemma3:4b
Successfully pulled gemma3:4b.
Attempting to pull model: llama3.2:1b
Successfully pulled llama3.2:1b.
Attempting to pull model: llama3.2:3b
Successfully pulled llama3.2:3b.


# Part 1: Direct AI Task Portfolio

In this part you complete four independent business tasks using the baseline model (`gemma3:1b`). Each task gives you a realistic business scenario and a source text. Your job is to engineer the prompt that produces the most useful output.

**How each task section works:**

1. **Write a zero-shot instruction** in the first code cell. Zero-shot means plain directions only — no examples, no step-by-step reasoning prompts. The code comment already labels it for you.
2. **Run the cell and preserve the output.** Do not delete or re-run the initial output before recording it. Your submission must show the original output.
3. **Diagnose the output** in the markdown cell that follows. Compare what the model produced against the source text. Identify one specific, evidence-based weakness (something missing, wrong, or poorly formatted). Quote or paraphrase both the source and the output.
4. **Write a revised instruction** in the second code cell, addressing the weakness you identified. For **at least two of the four tasks**, apply a named prompting strategy and identify it at the top of your instruction string.
5. **Evaluate the revision** in the final markdown cell. Explain whether the revision materially improved the output, what role the chosen strategy played, and what decisions still require a human's judgment.

**Named strategies you may apply for revised instructions:**

| Strategy | What it means |
|---|---|
| **Few-shot** | Include one or more examples of the expected input/output pattern before the task. |
| **Chain-of-thought** | Instruct the model to reason step by step before giving its final answer. |
| **Persona** | Assign the model a specific role or professional background before the task. |
| **Zero-shot** | Plain directions only — acceptable when the initial output already meets your standards. |

Preserve every initial output before revising an instruction.

In [9]:
PART1_SOURCES = {
  "extraction": "From: Maya Chen\nTo: Facilities Service Desk\nSubject: Loose handrail before Friday tour\n\nThe handrail in the east stairwell on floor 3 of Pioneer Hall is loose at the\nlower wall bracket. I noticed it at 9:15 a.m. on September 2. No one has been\ninjured, and the stairwell is still open. Please repair it before the visitor\ntour this Friday if possible. I can meet a technician after 1:00 p.m. Call me\nat extension 5521.",
  "summarization": "Customer Elena Ruiz reported that order OR-8841 was charged twice. The\noriginal $186.40 charge posted on August 6, and a second $186.40 charge posted\non August 8 after she refreshed the checkout page. The order itself arrived on\nAugust 10 and was correct. Agent Malik opened case CS-2197 on August 11 and\nasked Billing to investigate. Billing has not yet confirmed whether the second\nentry is a settled charge or a temporary authorization. Elena wants the second\ncharge removed if it settled, but she does not want the order canceled. She\nasked for an update by August 13 because her card payment is due August 14.",
  "drafting": "Supplier Northstar Filtration notified Procurement that shipment NF-771,\ncontaining 12 replacement filters, will arrive August 19 instead of August 14.\nThe plant currently has approximately four days of filter inventory at normal\nusage. Northstar offered expedited shipping for an additional fee, but\nProcurement has not approved that option. Operations is checking whether usage\ncan be reduced safely. The plant manager needs a status update today. No\nproduction shutdown has been scheduled.",
  "classification": "Routing categories:\n- IT Support: computers, software, networks, and accounts\n- Security Access: badges, controlled doors, and physical-access permissions\n- Facilities: building fixtures, utilities, and room conditions\n\nUrgency rules:\n- Urgent: an active safety issue or current business operation is blocked with\n  no workaround\n- Standard: future need, routine repair, or a workable temporary alternative\n\nRequest: \"My new analyst starts Monday. Her employee account works, but her\nbadge does not open the Finance Annex. I can meet her in the lobby and escort\nher on the first day if needed. Please add normal weekday access before 8:00\na.m. Monday. The request does not include the analyst's employee ID or the\nmanager's access approval record.\" "
}


## Part 1.1: Extraction

In this section you are fulfilling the role of an AI Automation Specialist working as a consultant for a local facilities management operation. They have been struggling with the amount of time it takes to read and synthesize information from unstructured emails from customers at various facilities. Your job is to extract key items from emails.

Your first prompt must be a **ZERO-SHOT PROMPT**. In the next section you will use other strategies to improve the output.

**Business user:** Facilities coordinator  
**Purpose:** Turn an emailed repair request into a consistent intake record.

**Provided input**

```text
From: Maya Chen
To: Facilities Service Desk
Subject: Loose handrail before Friday tour

The handrail in the east stairwell on floor 3 of Pioneer Hall is loose at the
lower wall bracket. I noticed it at 9:15 a.m. on September 2. No one has been
injured, and the stairwell is still open. Please repair it before the visitor
tour this Friday if possible. I can meet a technician after 1:00 p.m. Call me
at extension 5521.
```


#### TODO - INSTRUCT 🔧

In [10]:
# 🔧 TODO - INSTRUCT
# Strategy: zero-shot — clear directions only, no examples.
# Your GOAL here is to extract the following pieces of information from the email.
# Extract reporter name, location, problem description, date and time observed, urgency or deadline, and contact information."

initial_instruction_extraction = """

Read the email below and extract the following information:

- Reporter name
- Location
- Problem description
- Date and time observed
- Urgency or deadline
- Contact information

List each item on its own line, labeled clearly. If any piece of information is not present in the email, write "Not provided" for that field instead of guessing.

Email:
{email_text}
"""

In [11]:
# Run this to generate output after updating your instruction. It will take at least 20 seconds to run each loop.

for i in range(3):

  initial_record_extraction = run_part1(
      task_key="extraction",
      stage="initial",
      instruction=initial_instruction_extraction,
      source=PART1_SOURCES["extraction"],
  )
  display_record(initial_record_extraction)

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-08T18:53:14+00:00  
**Elapsed:** 11.11 seconds

Here’s the extracted information:

**Reporter name:** Maya Chen
**Location:** Pioneer Hall, east stairwell on floor 3
**Problem description:** A loose handrail exists at the lower wall bracket in the east stairwell.
**Date and time observed:** September 2, 2024 at 9:15 a.m. 
**Urgency/Deadline:** Not provided - It requires immediate repair prior to the visitor tour this Friday.
**Contact information:** Extension 5521

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-08T18:53:25+00:00  
**Elapsed:** 9.38 seconds

- Reporter name: Maya Chen
- Location: Pioneer Hall, east stairwell on floor 3
- Problem description: The handrail in the lower wall bracket is loose at the east stairwell.
- Date and time observed: 9:15 a.m. on September 2, 2024
- Urgency or deadline: Please repair the handrail before the visitor tour this Friday if possible.
- Contact information: Extension 5521

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-08T18:53:34+00:00  
**Elapsed:** 10.19 seconds

* **Reporter Name:** Maya Chen
* **Location:** Pioneer Hall - East stairwell on floor 3
* **Problem Description:** The handrail in the east stairwell on floor 3 is loose at the lower wall bracket.
*   **Date & Time Observed:** 9:15 a.m. on September 2nd.
* **Urgency/Deadline:** Requires repair before the visitor tour this Friday, but not explicitly stated.
*   **Contact Information:** Extension 5521

### TODO - REFLECT 🖊

Initial-Output Diagnosis and Revision Plan



**Specific weakness in the initial output:**

The date changes between runs even though the source email never changed.
Run 1: September 2 Run 2: August 26, 2023
Run 3: September 2, 2023 A date field should be copied exactly from the email every time not the model guessing.

**Speed of Output:**

CPU average: 10.55 seconds

GPU average: using TPU

**Planned instruction change:**

Tell the model to copy the date/time exactly as written in the email and
never add a year or change any digit if one isn't stated.

**Prompting strategy for the revision:**

Chain-of-thought. Have the model first quote the exact date/time phrase
from the email, then use that quote to fill the field.

### Revise your prompt with a strategy to address the issue you noted above with the model output quality.

### TODO - INSTRUCT 🔧

In [14]:
# 🔧 TODO. Write the revised instruction.
revised_instruction_extraction = """

Read the email below. For each item, first find the exact words in the email that give that information, then write the final answer based only on those exact words. Do not add, guess, or complete any information that is not written in the email.

For the date and time especially: copy it exactly as written. Do not add a year or change any number if the email does not include it.

Work through it like this for each field:
1. Quote the exact phrase from the email.
2. Write the final labeled answer based on that quote.

Extract these fields, each as a single labeled line (no sub-bullets):
- Reporter name
- Location
- Problem description
- Date and time observed
- Urgency or deadline
- Contact information

If a field is not in the email, write "Not provided."

Email:
{email_text}
"""


In [15]:
# run this to create the output. It will take at least 20 seconds to run each loop.
for i in range(3):

  revised_record_extraction = run_part1(
      task_key="extraction",
      stage="revised",
      instruction=revised_instruction_extraction,
      source=PART1_SOURCES["extraction"],
  )
  display_record(revised_record_extraction)

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-08T18:55:07+00:00  
**Elapsed:** 11.38 seconds

Here's the requested information extracted from the provided email:

- Reporter name: Maya Chen
- Location: Pioneer Hall, east stairwell on floor 3
- Problem description: The handrail in the east stairwell on floor 3 of Pioneer Hall is loose at the lower wall bracket.
- Date and time observed: September 2, 9:15 a.m.
- Urognor or deadline: Please repair it before the visitor tour this Friday if possible.
- Contact information: Extension 5521

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-08T18:55:19+00:00  
**Elapsed:** 10.29 seconds

Here’s a breakdown of the email’s relevant information according to your guidelines:

- Reporter name: Maya Chen
- Location: Pioneer Hall
- Problem description: Loose handrail in east stairwell on floor 3
- Date and time observed: September 2, 9:15 a.m.
- Urgency or deadline: Please repair it before the visitor tour this Friday
- Contact information: Extension 5521

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-08T18:55:29+00:00  
**Elapsed:** 11.6 seconds

Here's the breakdown of the email according to your criteria:

- Reporter name: Maya Chen
- Location: Pioneer Hall, east stairwell
- Problem description: The handrail in the east stairwell on floor 3 of Pioneer Hall is loose at the lower wall bracket.
- Date and time observed: September 2, 9:15 a.m.
- Urgency or deadline: Please repair it before the visitor tour this Friday if possible.
- Contact information: Extension 5521

### TODO - REFLECT 🖊

Improvement and Human Review

**Effect of the revision:**

Yes, the date time is now identical across all 3 runs, and formatting is consistent which fixes the main issues from the first prompt.

**Role of the prompting strategy:**

The chain-of-thought "quote first, then answer" step forced the model to stop guessing and actually reference the email.

**Human review still required:**

A person should double check the location detail and
verify contact info and deadline wording.

**Output Variability**

It didnt change much. Only the location field changed between runs.

## Part 1.2: Summarization

**Business user:** Customer-service supervisor  
**Purpose:** Prepare a concise escalation summary without losing financial details.

**Provided input**

```text
Customer Elena Ruiz reported that order OR-8841 was charged twice. The
original $186.40 charge posted on August 6, and a second $186.40 charge posted
on August 8 after she refreshed the checkout page. The order itself arrived on
August 10 and was correct. Agent Malik opened case CS-2197 on August 11 and
asked Billing to investigate. Billing has not yet confirmed whether the second
entry is a settled charge or a temporary authorization. Elena wants the second
charge removed if it settled, but she does not want the order canceled. She
asked for an update by August 13 because her card payment is due August 14.
```


**Your task:** A customer-service supervisor needs a concise escalation summary to hand off to a billing team. The model should condense the case above without losing any financially important detail — amounts, dates, case numbers, and the customer's stated deadline all matter.

**What to do in the cells below:**
- **First code cell:** Replace the `TODO` with your zero-shot instruction. Specify the audience (the billing team), the required level of detail, and any format constraints (length, structure). Run the cell and leave the output visible.
- **Diagnosis markdown cell:** Compare the output against the source. Identify one specific weakness — for example, a missing amount, a dropped date, or a format that buries the urgency.
- **Second code cell:** Write your revised instruction addressing that weakness. Remember: at least two of your four tasks must apply a named strategy with an explanation.
- **Evaluation markdown cell:** Explain what improved, what the strategy contributed, and which facts in the summary would need human verification before the billing team acts on them.

### TODO - INSTRUCT 🔧

In [16]:
# 🔧 Strategy: zero-shot — clear directions only, no examples.
initial_instruction_summarization = """

You are preparing a summary for the billing team. Summarize the case below in 4-6 sentences.

Include every dollar amount, every date, the order number, the case number, and the customer's requested deadline. Do not omit any financial detail. Clearly state what the customer wants done.

Case:
{case_text}
"""

In [17]:
for i in range(3):

  initial_record_summarization = run_part1(
      task_key="summarization",
      stage="initial",
      instruction=initial_instruction_summarization,
      source=PART1_SOURCES["summarization"],
  )
  display_record(initial_record_summarization)

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-08T19:02:35+00:00  
**Elapsed:** 14.81 seconds

Here’s a 6-sentence summary suitable for the billing team briefing:

Elena Ruiz has contacted the billing team reporting two instances of an added charge – $186.40 – on her payment on August 8.  The second charge, appearing shortly after a refresh, further corroborates this issue. The original charge, dated August 6, is currently unexplained, pending investigation. The order number OR-8841 was originally authorized for $186.40. The current billing charge is $186.40 and the client seeks to have this charge removed without processing a refund of the original transaction. Billing requires confirming whether the second charge represents a settled payment or a temporary authorization to ensure accuracy and a timely resolution to her payment due date of August 14th.

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-08T19:02:50+00:00  
**Elapsed:** 17.48 seconds

Okay, here’s a summary of Elena Ruiz’s billing issue, suitable for the billing team’s perspective, incorporating all required detail and reaching the requested request deadline: 

**Subject: Outstanding Charge and Requested Adjustment - Case CS-2197**

Elena Ruiz notified the billing team of a duplicate charge on order OR-8841 totaling $186.40, appearing both on August 6th and August 8th.  Billing needs to confirm the nature of this duplicate transaction – it was initially applied as a settlement, but Elena requested that the second charge be removed upon confirmation that it represents a settled debit.  She specifically requests an update within 13 days, August 13th, regarding the status of this charge and a definitive resolution regarding the duplicate entry.  Failure to fulfill this request will necessitate further investigation and a review of associated transactions. 

---
**Note:**

I’ve left a lot of detail as you requested.  If the case text is significantly more complex, the summary could be adjusted to prioritize crucial elements.

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-08T19:03:08+00:00  
**Elapsed:** 12.89 seconds

Here's a four-sentence summary of the case:

Elena Ruiz notified the Billing Team on August 6, 2024 that two duplicate charge entries were posted on her account, OR-8841. Initially charged a $186.40, subsequently a $186.40 on August 8 after a refresh of the checkout page. The order itself arrived on August 10, with no issues reported. Billing is currently investigating the discrepancy and awaiting further confirmation regarding whether the second charge is a settled amount or a temporary authorization.  Mrs. Ruiz requests an update by August 13 to ensure her card payment is due on August 14th to resolve the issue.

### TODO - REFLECT 🖊

Initial-Output Diagnosis and Revision Plan

Use Strategy: zero-shot — clear directions only, no examples.

**Specific weakness in the initial output:**

Run 1 was repeated text and Run 2 misstated the deadline as 13 days instead of saying August 13. Run 3 invented a year not in the source.




**Planned instruction change:**

how the model an example of the correct clean format and tell it to copy
dates exactly as written, using few-shot as the revision strategy

**Prompting strategy for the revision:**

Few-shot because it shows the model the exact pattern to copy

**Your task:** A procurement analyst needs to send the plant manager a status update about a delayed shipment. The draft must be factually accurate, avoid making commitments that have not been approved (e.g., expedited shipping has not been authorized), and convey appropriate urgency without overstating the risk.

**What to do in the cells below:**
- **First code cell:** Replace the `TODO` with your zero-shot instruction. Specify the intended recipient (the plant manager), the tone, and any constraints on what the draft should or should not commit to. Run the cell and leave the output visible.
- **Diagnosis markdown cell:** Review the draft for any facts that differ from the source, commitments the model made that are not supported, or tone and structure problems.
- **Second code cell:** Write your revised instruction. Consider whether a persona strategy (e.g., "You are a procurement analyst...") or chain-of-thought reasoning helps the model avoid unsupported commitments.
- **Evaluation markdown cell:** Explain what improved, what the strategy contributed, and what a human analyst must check before sending the draft.

### Revise your prompt with a strategy to address the issue you noted above with the model output quality.

### TODO - INSTRUCT 🔧

In [20]:
# 🔧 Identify your strategy at the start of the instruction string, e.g.: "Strategy: persona"
revised_instruction_summarization = """

You are drafting a status update for the plant manager about a delayed shipment. Write 4-6 sentences in a professional, factual tone.

Only state facts that are explicitly given below. Do not say expedited shipping has been approved, and do not say a shutdown has been scheduled, unless the source explicitly says so. Convey appropriate urgency without exaggerating the risk.

Source:
{shipment_text}
"""

In [21]:
for i in range(3):

  revised_record_summarization = run_part1(
      task_key="summarization",
      stage="revised",
      instruction=revised_instruction_summarization,
      source=PART1_SOURCES["summarization"],
  )
  display_record(revised_record_summarization)

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-08T19:34:32+00:00  
**Elapsed:** 13.5 seconds

Here's a draft for the status update:

"Order OR-8841 experienced a slight delay in shipment, as reported by Customer Elena Ruiz. The original charge posted on August 6th was subsequently updated to $186.40 on August 8th, while a second charge for $186.40 was posted following a refresh on August 10th. The order arrives on August 10th with no noted issues, and billing has established a case CS-2197 for investigation. Customer Ruiz requested removal of the second charge to reflect a settled payment.  We anticipate resolving this issue and reviewing the billing records by August 13th."

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-08T19:34:46+00:00  
**Elapsed:** 12.09 seconds

Okay, here’s a status update for the Plant Manager, sticking to the provided text:

"The Shipment OR-8841 is experiencing an anomaly resulting in two duplicate charges. Originally charged on August 6th and then August 8th, the order arrived August 10th with accurate data. Agent Malik initiated a case (CS-2197) regarding this discrepancy, requesting investigation by Billing. Customer Elena Ruiz has requested removal of the second charge due to an accrued amount. We are diligently monitoring this situation, aiming to provide an update by August 13th to assess the resolution."

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-08T19:34:58+00:00  
**Elapsed:** 12.09 seconds

Here’s a draft status update:

“The plant manager has been informed that there have been two duplicate charges posted for order OR-8841, resulting in a second $186.40 charge.”

“Customer Elena Ruiz is contacting customer service regarding this issue, and billing has yet to complete investigation.”

“Payment processing for the customer’s card is approaching its due date and requires immediate attention.”

“The Plant Manager has been tracking the situation, and we are monitoring closely to ensure a resolution is reached as swiftly as possible to mitigate any operational disruption as outlined in Elena’s requests.”

### TODO - REFLECT 🖊

Improvement and Human Review

**Effect of the revision:**

Not much improvement because the drafts still reference billing-case detailsrather than staying fully on the shipment update
**Role of the prompting strategy:**

The persona/zero-shot instruction kept the tone professional, but it didn't fully stop the content drift from the source.
**Human review still required:**

A human must verify every fact and date before sending, remove any unsupported claims, and confirm no unapproved commitments

## Part 1.3: Drafting

**Business user:** Procurement analyst  
**Purpose:** Draft an internal delay notice that does not make unsupported commitments.

**Provided input**

```text
Supplier Northstar Filtration notified Procurement that shipment NF-771,
containing 12 replacement filters, will arrive August 19 instead of August 14.
The plant currently has approximately four days of filter inventory at normal
usage. Northstar offered expedited shipping for an additional fee, but
Procurement has not approved that option. Operations is checking whether usage
can be reduced safely. The plant manager needs a status update today. No
production shutdown has been scheduled.
```


**Your task:** A service-desk dispatcher needs to route this access request to the correct team and assign the correct urgency level. The routing and urgency definitions are included in the source text above — the model should apply them, not invent its own categories. The model should also flag that required information (employee ID and approval record) is missing.

**What to do in the cells below:**
- **First code cell:** Replace the `TODO` with your zero-shot instruction. Tell the model to output the routing category, urgency level, and any missing information that blocks processing, based strictly on the definitions provided. Run the cell and leave the output visible.
- **Diagnosis markdown cell:** Check whether the model applied the urgency definitions correctly (does the workaround change the urgency?), used only the defined categories, and flagged the missing fields. Identify the most significant gap.
- **Second code cell:** Write your revised instruction. Chain-of-thought is often effective here — prompting the model to reason through each definition before giving a final answer can reduce misclassification.
- **Evaluation markdown cell:** Explain what improved, what the strategy contributed, and what a human dispatcher must still decide before acting on the model's output.

### TODO - INSTRUCT 🔧

In [22]:
# 🔧 TODO - INSTRUCT
# Strategy: zero-shot — clear directions only, no examples.
initial_instruction_drafting = """

You are a procurement analyst writing a status update for the plant manager about a delayed shipment. Write 4-6 sentences in a professional, factual tone.

Only include facts explicitly stated in the source below. Do not say expedited shipping has been approved. Do not say a production shutdown has been scheduled. Convey appropriate urgency based on current inventory levels without exaggerating the risk.

Source:
{shipment_text}
"""

In [23]:
# Run this to generate output after updating your instruction. It will take at least 20 seconds to run each loop.

for i in range(3):
  initial_record_drafting = run_part1(
      task_key="drafting",
      stage="initial",
      instruction=initial_instruction_drafting,
      source=PART1_SOURCES["drafting"],
  )
  display_record(initial_record_drafting)

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-08T19:40:05+00:00  
**Elapsed:** 10.7 seconds

Subject: Urgent Shipment Delay - NF-771

Good Morning,

Due to notification from Northstar Filtration, shipment NF-771, a supply of 12 replacement filters, is experiencing a delay. The shipment is expected to arrive August 19th, instead of August 14th, resulting in a gap of four days for filter usage. Operations is evaluating potential methods to minimize the impact on ongoing manufacturing operations. Due to this delay and current inventory levels, we are monitoring utilization closely and will continue to review adjustments needed for optimal performance.

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-08T19:40:16+00:00  
**Elapsed:** 11.99 seconds

Here's a draft status update, meeting your specified requests:

“Good Morning, Manager. Supplier Northstar Filtration has provided notification that shipment NF-771, containing 12 replacement filters, will be delivered August 19, 2024, instead of August 14th as initially planned.”  “Due to this revised shipping date, we currently have approximately four days of filter inventory remaining at standard usage levels.” “Northstar Filtration initially suggested expedited shipping at an added cost but we have not yet received a formal approval.” “Operations is reviewing current usage patterns to assess whether further measures can be implemented to minimize potential inventory impact.” "I’ve compiled the status update for your review now."

**Model:** `gemma3:1b`  
**Recorded:** 2026-09-08T19:40:28+00:00  
**Elapsed:** 9.7 seconds

Subject: Shipment Delay - NF-771 Filter Supply

This report provides a status update regarding shipment NF-771, which contains 12 replacement filters currently under review for usage. Supplier Northstar Filtration has informed us that the shipment will now arrive August 19th, instead of August 14th. Despite a previously agreed upon expedited shipping arrangement, procurement has not authorized this option. Ongoing assessment within operations is evaluating whether decreased usage can mitigate potential inventory challenges in the meantime.  Full detail regarding anticipated usage impact is available in the attached documentation.

### TODO - REFLECT 🖊

Initial-Output Diagnosis and Revision Plan

**Specific weakness in the initial output:**

Run 2 invents a year (2024) not in the source, and none of the three
drafts explicitly confirm no shutdown has been scheduled even though
the source states this directly.

**Planned instruction change:**

Require the model to explicitly state the shutdown and expedited-shipping
status using the source's own wording, and forbid adding any date details
not in the source.

**Prompting strategy for the revision:**

Chain-of-thought — have the model list the key facts first, then write
the draft, which should reduce omissions and invented details.

### Revise your prompt with a strategy to address the issue you noted above with the model output quality.

### TODO - INSTRUCT 🔧

In [1]:
# 🔧 TODO - INSTRUCT
# Identify your strategy at the start of the instruction string, e.g.: "Strategy: chain-of-thought"
revised_instruction_drafting = """

Strategy: chain-of-thought

You are a procurement analyst writing a status update for the plant manager about a delayed shipment.

First, list these key facts exactly as stated in the source below, with no additions:
1. New arrival date (exact, no year unless the source gives one)
2. Days of filter inventory remaining
3. Whether expedited shipping has been approved
4. Whether a production shutdown has been scheduled

Then, using only those listed facts, write a 4-6 sentence professional status update for the plant manager. You must explicitly state the shutdown status and the expedited shipping status using wording close to the source. Do not add any fact, date, or approval not listed above.

Source:
{shipment_text}
"""

In [2]:
# Run this to generate output after updating your instruction.

revised_record_drafting = run_part1(
    task_key="drafting",
    stage="revised",
    instruction=revised_instruction_drafting,
    source=PART1_SOURCES["drafting"],
)
display_record(revised_record_drafting)

NameError: name 'run_part1' is not defined

### TODO - REFLECT 🖊

Improvement and Human Review

**Effect of the revision:**

🖊 TODO: Explain whether the revision materially improved the output and why.

**Role of the prompting strategy:**

🖊 TODO: Explain what the named strategy contributed — or, if you stayed with zero-shot, explain why no examples, reasoning steps, or persona were needed.

**Human review still required:**

🖊 TODO: Identify decisions, facts, risks, or actions that remain a person's responsibility.

# Part 2: Controlled Four-Model Evaluation

In Part 1, you were free to revise your instructions and iterate. Part 2 is different: you write **one** instruction and send it to all four models **unchanged**. This is a controlled experiment — the only variable is the model itself.

**Why controlled?** If you give different prompts to different models, any differences in output could come from your instruction, not from the model. A shared, unmodified prompt isolates the model as the only variable and makes your comparisons valid.



## Service Issue Details

```text
REQUEST SR-2401
Site: North Distribution Center
Reported by: Luis Ortega, extension 4410
At 6:40 a.m. on June 12, the quality-control freezer display read 18 F. Its
required operating range is 0-5 F. Temperature-sensitive calibration
materials were moved to the backup freezer. Staff reset the alarm twice,
but it returned both times. No employee injury was reported.
```


In [ ]:
multimodeltext = """
REQUEST SR-2401
Site: North Distribution Center
Reported by: Luis Ortega, extension 4410
At 6:40 a.m. on June 12, the quality-control freezer display read 18 F. Its
required operating range is 0-5 F. Temperature-sensitive calibration
materials were moved to the backup freezer. Staff reset the alarm twice,
but it returned both times. No employee injury was reported.
"""

### TODO - INSTRUCT 🔧

In [ ]:
# 🔧 TODO - INSTRUCT
# Strategy: zero-shot — clear directions only, no examples.
initial_instruction_extraction = """

TODO: Use your updated summarization instruction from Part 1.1 here

"""



In [ ]:
# Run this after revising the

comparison_prompt = compose_prompt(
    initial_instruction_extraction,
    multimodeltext,
)

comparison_records = {}
for model in REQUIRED_MODELS:
    print(f"Running independent comparison: {model}")
    comparison_records[model] = chat_once(
        model,
        comparison_prompt,
        f"comparison_{model}",
    )
    display_record(comparison_records[model])


### TODO - REFLECT 🖊

Model Evaluation

**How similar or different were the outputs?:**

🖊 TODO: Cite evidence from the output.

**How long did each model take to run?:**

🖊 TODO: How fast or slow were each of the models? Do you notice a pattern for models that ran faster vs slower?

**Which model was "Best"?:**

🖊 TODO: Are the model differences substantial enough to merit choosing a particular model? If so, which?

# Part 3: Reflect on the Assignment



#### TODO - FINAL REFLECTION 🖊

Write a **150–250 word** reflection and answer the

1. **Prompt effect (Part 1):** Which revision across your four Part 1 tasks produced the largest improvement? What specifically changed in the model's output, and why did the revised instruction work better?

🖊 TODO: YOUR ANSWER HERE

2. **Strategy fit:** For each named strategy you applied (few-shot, chain-of-thought, or persona), explain why you chose it for that task. If you kept zero-shot for any revision, explain why plain directions were sufficient.

🖊 TODO: YOUR ANSWER HERE

3. **Model differences (Part 2):** Where did the four models diverge most noticeably — in accuracy, completeness, format adherence, or something else? Give a specific example.

🖊 TODO: YOUR ANSWER HERE

4. **Size versus quality:** Did the larger model in each family (Gemma 3 4B, Llama 3.2 3B) consistently produce better outputs than the smaller one? Were there cases where the size difference did not predict quality?

🖊 TODO: YOUR ANSWER HERE

5. **Runtime trade-offs:** How did elapsed time vary across the four models? Was the quality gain from the slower models worth the additional time, given the nature of this task?

🖊 TODO: YOUR ANSWER HERE

6. **Business risk:** Identify one specific output — from either Part 1 or Part 2 — that would cause a real problem if a person acted on it without review. What is the specific risk, and what kind of human check would catch it?

🖊 TODO: YOUR ANSWER HERE

7. **Generalization:** Based on what you observed, under what conditions would a small local model like these be a reasonable choice for a business task? Under what conditions would you want a larger or cloud-hosted model instead?